In [ ]:
from datasets import load_dataset
import gensim.downloader
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
dataset = load_dataset("google-research-datasets/go_emotions", "simplified")

Using the latest cached version of the dataset since google-research-datasets/go_emotions couldn't be found on the Hugging Face Hub


Found the latest cached dataset configuration 'simplified' at C:\Users\temy2\.cache\huggingface\datasets\google-research-datasets___go_emotions\simplified\0.0.0\add492243ff905527e67aeb8b80c082af02207c3 (last modified on Tue Apr 28 02:13:06 2026).


In [ ]:
import spacy

class EnglishTextNormalizer:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_md", disable=["ner", "parser"])
        self.nlp.add_pipe("sentencizer", before="tok2vec")

    def __call__(self, texts_list):
        for doc in self.nlp.pipe(texts_list, batch_size=512):
            for sentence in doc.sents:
                clean_sentence = []
                for token in sentence:
                    if (
                        # not token.is_stop      and 
                        not token.is_punct 
                        and token.is_alpha 
                    ):
                        lemma = token.lemma_.lower()
                        clean_sentence.append(lemma)
                
                if clean_sentence:
                    yield clean_sentence


In [28]:
dataset['train'][0]['text']
normalizer = EnglishTextNormalizer()
train = [x['text'] for x in dataset['train']]
train

["My favourite food is anything I didn't have to cook myself.",
 'Now if he does off himself, everyone will think hes having a laugh screwing with people instead of actually dead',
 'WHY THE FUCK IS BAYLESS ISOING',
 'To make her feel threatened',
 'Dirty Southern Wankers',
 "OmG pEyToN iSn'T gOoD eNoUgH tO hElP uS iN tHe PlAyOfFs! Dumbass Broncos fans circa December 2015.",
 'Yes I heard abt the f bombs! That has to be why. Thanks for your reply:) until then hubby and I will anxiously wait 😝',
 'We need more boards and to create a bit more space for [NAME]. Then we’ll be good.',
 'Damn youtube and outrage drama is super lucrative for reddit',
 'It might be linked to the trust factor of your friend.',
 'Demographics? I don’t know anybody under 35 who has cable tv.',
 "Aww... she'll probably come around eventually, I'm sure she was just jealous of [NAME]... I mean, what woman wouldn't be! lol ",
 'Hello everyone. Im from Toronto as well. Can call and visit in personal if needed.',
 "R/s

In [29]:
res = list(normalizer(train))
res

[['my',
  'favourite',
  'food',
  'be',
  'anything',
  'i',
  'do',
  'have',
  'to',
  'cook',
  'myself'],
 ['now',
  'if',
  'he',
  'do',
  'off',
  'himself',
  'everyone',
  'will',
  'think',
  'he',
  's',
  'have',
  'a',
  'laugh',
  'screw',
  'with',
  'people',
  'instead',
  'of',
  'actually',
  'dead'],
 ['why', 'the', 'fuck', 'be', 'bayless', 'isoing'],
 ['to', 'make', 'she', 'feel', 'threaten'],
 ['dirty', 'southern', 'wanker'],
 ['omg',
  'peyton',
  'good',
  'enough',
  'to',
  'help',
  'we',
  'in',
  'the',
  'playoffs'],
 ['dumbass', 'broncos', 'fan', 'circa', 'december'],
 ['yes', 'i', 'hear', 'abt', 'the', 'f', 'bomb'],
 ['that', 'have', 'to', 'be', 'why'],
 ['thank',
  'for',
  'your',
  'reply',
  'until',
  'then',
  'hubby',
  'and',
  'i',
  'will',
  'anxiously',
  'wait'],
 ['we',
  'need',
  'more',
  'board',
  'and',
  'to',
  'create',
  'a',
  'bit',
  'more',
  'space',
  'for',
  'name'],
 ['then', 'we', 'be', 'good'],
 ['damn',
  'youtube',
 

In [37]:
from huggingface_hub import hf_hub_download
from gensim.models import KeyedVectors, Word2Vec

repo_id = "fse/word2vec-google-news-300"

model_path = hf_hub_download(repo_id=repo_id, filename="word2vec-google-news-300.model")
hf_hub_download(repo_id=repo_id, filename="word2vec-google-news-300.model.vectors.npy")


kv = KeyedVectors.load(model_path, mmap="r")
model = Word2Vec(
    vector_size=kv.vector_size,
    window=5,
    min_count=1,
    workers=4,
    sg=1,  
)


In [41]:
model.build_vocab(res)

for word, idx in model.wv.key_to_index.items():
    if word in kv:
        model.wv.vectors[idx] = kv[word]

model.train(res, total_examples=len(res), epochs = 5)
model.wv.save_word2vec_format('word2vec_emotional.bin', binary=True)
model.save('word2vec_emotional.model')

In [40]:
len(res)

65192